In [1]:
import pandas as pd
import numpy as np
import requests
import warnings

from sqlalchemy import create_engine,text, FLOAT, INTEGER, VARCHAR, Table, Column, MetaData

# 1. Delay BTS

In [2]:
df_bts_delay = pd.read_csv('C:/Users/Adrian Lambotte/Desktop/TFE/2 - Tables & Sources/Tables/bts_15_airports.csv', low_memory=False)
df_bts_delay.head(5)

,Year,Quarter,Month,DayofMonth,DayOfWeek,FlightDate,Reporting_Airline,DOT_ID_Reporting_Airline,IATA_CODE_Reporting_Airline,Tail_Number,...,CancellationCode,Diverted,CRSElapsedTime,ActualElapsedTime,Distance,CarrierDelay,WeatherDelay,NASDelay,SecurityDelay,LateAircraftDelay
0,2019,1,1,4,5,2019-01-04,OO,20304,OO,N679SA,...,NaN,0.0,115.0,93.0,524.0,NaN,NaN,NaN,NaN,NaN
1,2019,1,1,4,5,2019-01-04,OO,20304,OO,N145SY,...,NaN,0.0,124.0,129.0,641.0,NaN,NaN,NaN,NaN,NaN
2,2019,1,1,4,5,2019-01-04,OO,20304,OO,N213SY,...,NaN,0.0,54.0,39.0,73.0,NaN,NaN,NaN,NaN,NaN
3,2019,1,1,4,5,2019-01-04,OO,20304,OO,N975SW,...,NaN,0.0,62.0,53.0,109.0,NaN,NaN,NaN,NaN,NaN
4,2019,1,1,4,5,2019-01-04,OO,20304,OO,N937EV,...,NaN,0.0,127.0,115.0,639.0,NaN,NaN,NaN,NaN,NaN


### 1.1 Changing the data type

In [4]:
df_bts_delay['FlightDate'] = pd.to_datetime(df_bts_delay['FlightDate'])
df_bts_delay['Flight_Number_Reporting_Airline'] = df_bts_delay['Flight_Number_Reporting_Airline'].astype('Int64')
df_bts_delay['DepTime'] = df_bts_delay['DepTime'].astype('Int64')
df_bts_delay['DepDelay'] = df_bts_delay['DepDelay'].astype('Int64')
df_bts_delay['TaxiOut'] = df_bts_delay['TaxiOut'].astype('Int64')
df_bts_delay['WheelsOff'] = df_bts_delay['WheelsOff'].astype('Int64')
df_bts_delay['WheelsOn'] = df_bts_delay['WheelsOn'].astype('Int64')
df_bts_delay['TaxiIn'] = df_bts_delay['TaxiIn'].astype('Int64')
df_bts_delay['ArrTime'] = df_bts_delay['ArrTime'].astype('Int64')
df_bts_delay['ArrDelay'] = df_bts_delay['ArrDelay'].astype('Int64')
df_bts_delay['Cancelled'] = df_bts_delay['Cancelled'].astype(bool)
df_bts_delay['Diverted'] = df_bts_delay['Diverted'].astype(bool)
df_bts_delay['CRSElapsedTime'] = df_bts_delay['CRSElapsedTime'].astype('Int64')
df_bts_delay['ActualElapsedTime'] = df_bts_delay['ActualElapsedTime'].astype('Int64')
df_bts_delay['Distance'] = df_bts_delay['Distance'].astype('Int64')
df_bts_delay['CarrierDelay'] = df_bts_delay['CarrierDelay'].astype('Int64')
df_bts_delay['WeatherDelay'] = df_bts_delay['WeatherDelay'].astype('Int64')
df_bts_delay['NASDelay'] = df_bts_delay['NASDelay'].astype('Int64')
df_bts_delay['SecurityDelay'] = df_bts_delay['SecurityDelay'].astype('Int64')
df_bts_delay['LateAircraftDelay'] = df_bts_delay['LateAircraftDelay'].astype('Int64')

### 1.2 Column Engineering

In [ ]:
#Keeping only city name
df_bts_delay['OriginCityName'] = df_bts_delay['OriginCityName'].str.split(', ').str[0]
df_bts_delay['DestCityName'] = df_bts_delay['DestCityName'].str.split(', ').str[0]

#Setting negative delays to 0
df_bts_delay['DepDelayPositive'] = df_bts_delay['DepDelay'].clip(lower=0)
df_bts_delay['ArrDelayPositive'] = df_bts_delay['ArrDelay'].clip(lower=0)

#Setting positive delays to 0 to measure  the lead time of flights
df_bts_delay['DepEarly'] = df_bts_delay['DepDelay'].clip(upper=0).abs()
df_bts_delay['ArrEarly'] = df_bts_delay['ArrDelay'].clip(upper=0).abs()

#Converting distances from miles to kilometres
df_bts_delay['DistanceKm'] =  df_bts_delay['Distance'] * 1.60934

### 1.3 Exporting to SQL Server

In [ ]:
dbms = 'YOUR DBMS'
server = r'YOUR SERVER'
driver = 'YOUR DRIVER'
db_name = 'TFE_staging'

con_string = url = f'{dbms}://{server}/{db_name}?trusted_connection=yes&driver={driver}'

con = create_engine(con_string, fast_executemany = True)

In [ ]:
#Setting the type for SQL Server
dtype_map = {col: VARCHAR(100) for col in df_bts_delay.select_dtypes(include=['string', 'object']).columns}
dtype_map.update({col: INTEGER() for col in df_bts_delay.select_dtypes(include=['int64', 'Int64']).columns})

df_bts_delay.to_sql(name ='Delay', con = con, if_exists = 'replace', index = False, chunksize=10000, dtype=dtype_map)

-1052

# 2. Airport Size

In [ ]:
df_airport_size = pd.read_excel('YOUR FILE')
df_airport_size

,Rank,RO,ST,Locid,City,Airport Name,S/L,Hub,CY 24 Enplanements,CY 23 Enplanements,% Change
0,1.0,SO,GA,ATL,Atlanta,Hartsfield/Jackson Atlanta International,P,L,52511402.0,50950068.0,0.0306
1,2.0,SW,TX,DFW,Fort Worth,Dallas-Fort Worth International,P,L,42351316.0,39246212.0,0.0791
2,3.0,NM,CO,DEN,Denver,Denver International,P,L,40012895.0,37863967.0,0.0568
3,4.0,GL,IL,ORD,Chicago,Chicago O'Hare International,P,L,38575693.0,35843104.0,0.0762
4,5.0,WP,CA,LAX,Los Angeles,Los Angeles International,P,L,37760834.0,40956673.0,-0.0780
...,...,...,...,...,...,...,...,...,...,...,...
515,561.0,AL,AK,GGV,Kwigillingok,Kwigillingok,CS,NaN,2534.0,2241.0,0.1307
516,562.0,AL,AK,KLG,Upper Kalskag (native name: Kalskag),Kalskag,CS,NaN,2524.0,2029.0,0.2440
517,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
518,NaN,NaN,NaN,119,Nonprimary Commercial Service,NaN,NaN,NaN,NaN,NaN,NaN


In [5]:
#Keeping only the 15 airports
airports = ["ATL", "DFW", "DEN", "LAX", "JFK", "PDX", "RDU", "STL", "MSY", "RSW", "RNO", "OKC", "GRR", "BZN", "PWM"]

df_airport_size = df_airport_size[df_airport_size['Locid'].isin(airports)].reset_index()

### 2.1 Changing the data type

In [14]:
df_airport_size['Rank'] = df_airport_size['Rank'].astype('int')
df_airport_size['CY 24 Enplanements'] = df_airport_size['CY 24 Enplanements'].astype('int')
df_airport_size['CY 23 Enplanements'] = df_airport_size['CY 23 Enplanements'].astype('int')
df_airport_size['Locid'] = df_airport_size['Locid'].astype('str')
df_airport_size['% Change'] = df_airport_size['% Change'] * 100


### 2.2 Exporting to SQL Server

In [15]:
dtype_map = {col: VARCHAR(100) for col in df_airport_size.select_dtypes(include='string').columns}
dtype_map.update({col: INTEGER() for col in df_airport_size.select_dtypes(include='int64').columns})

df_airport_size.to_sql(name ='airport_size', con = con, if_exists = 'replace', index = False, dtype= dtype_map)

-1

# 3. Runways Airport

In [ ]:
df_runways_airport = pd.read_excel('YOUR FILE')
df_runways_airport.head(5)

,OID,EFF_DATE,SITE_NO,SITE_TYPE_CODE,STATE_CODE,ARPT_ID,ARPT_NAME,CITY,COUNTRY_CODE,OWNERSHIP_TYPE_CODE,...,DTRM_METHOD_CODE,RWY_LGT_CODE,RWY_LEN_SOURCE,LENGTH_SOURCE_DATE,GROSS_WT_SW,GROSS_WT_DW,GROSS_WT_DTW,GROSS_WT_DDTW,Shape__Area,Shape__Length
0,1,2026/04/16,103.0,A,AL,0J0,ABBEVILLE MUNI,ABBEVILLE,US,PU,...,NaN,MED,3RD PARTY SURVEY,2025/06/17,NaN,NaN,NaN,NaN,48174.937500,3646.377235
1,2,2026/04/16,106.0,A,AL,2A8,ADDISON MUNI,ADDISON,US,PU,...,NaN,NaN,OWNER,2011/08/08,NaN,NaN,NaN,NaN,40336.621094,2033.825139
2,3,2026/04/16,110.0,A,AL,EET,SHELBY COUNTY,ALABASTER,US,PU,...,NaN,MED,STATE,2006/06/13,16.0,NaN,NaN,NaN,49867.644531,3707.318451
3,4,2026/04/16,112.1,A,AL,BFZ,ALBERTVILLE RGNL/THOMAS J BRUMLIK FLD,ALBERTVILLE,US,PU,...,NaN,MED,3RD PARTY SURVEY,2024/11/02,60.0,90.0,130.0,NaN,83300.039062,4586.225741
4,5,2026/04/16,116.0,A,AL,ALX,THOMAS C RUSSELL FLD,ALEXANDER CITY,US,PU,...,NaN,MED,3RD PARTY SURVEY,2010/01/19,30.0,NaN,NaN,NaN,68799.781250,4021.180841


In [ ]:
#Selecting the 15 airports and keeping the useful columns
df_runways_airport = df_runways_airport[['ARPT_ID', 'ARPT_NAME', 'CITY', 'RWY_ID','RWY_LEN', 'RWY_WIDTH', 'SURFACE_TYPE_CODE', 'COND', 'RWY_LGT_CODE', 'RWY_LEN_SOURCE']]

airports = ["ATL", "DFW", "DEN", "LAX", "JFK", "PDX", "RDU", "STL", "MSY", "RSW", "RNO", "OKC", "GRR", "BZN", "PWM"]
df_runways_airport = df_runways_airport[df_runways_airport['ARPT_ID'].isin(airports)]

In [18]:
df_runways_airport['RWY_LEN_M'] = df_runways_airport['RWY_LEN'] * 0.3048

### 3.1 Changing the data type

In [ ]:
df_runways_airport.dtypes
#I don't need to change the data type

ARPT_ID                  str
ARPT_NAME                str
CITY                     str
RWY_ID                   str
RWY_LEN                int64
RWY_WIDTH              int64
SURFACE_TYPE_CODE        str
COND                     str
RWY_LGT_CODE             str
RWY_LEN_SOURCE           str
RWY_LEN_M            float64
dtype: object

### 3.2 Exporting to SQL Server

In [20]:
dtype_map = {col: VARCHAR(100) for col in df_runways_airport.select_dtypes(include='string').columns}
dtype_map.update({col: INTEGER() for col in df_runways_airport.select_dtypes(include='int64').columns})

df_runways_airport.to_sql(name ='runways_airport', con = con, if_exists = 'replace', index = False, dtype= dtype_map)

-1

# 4. Demographics

In [ ]:
url = "https://api.census.gov/data/2022/acs/acs5"
params = {
    "get": "NAME,B01003_001E,B19013_001E,B15003_001E,B15003_022E",
    "for": "metropolitan statistical area/micropolitan statistical area:*",
    "key": "YOUR KEY"
}

response = requests.get(url, params=params)
data = response.json()

df_demographics = pd.DataFrame(data[1:], columns=data[0])

# Rename column
df_demographics = df_demographics.rename(columns={
    "B01003_001E": "population",
    "B19013_001E": "median_household_income",
    "B15003_001E": "pop_25plus",
    "B15003_022E": "bachelors_count"
})

# Converting the type into numerics
for col in ["population", "median_household_income", "pop_25plus", "bachelors_count"]:
    df_demographics[col] = pd.to_numeric(df_demographics[col], errors="coerce")

# Calculating the percentage of bachelors
df_demographics["pct_bachelors"] = (df_demographics["bachelors_count"] / df_demographics["pop_25plus"] * 100).round(1)

df_demographics.head(5)

,NAME,population,median_household_income,pop_25plus,bachelors_count,metropolitan statistical area/micropolitan statistical area,pct_bachelors
0,"Aberdeen, SD Micro Area",42292,70693,28243,6728,10100,23.8
1,"Aberdeen, WA Micro Area",75672,59105,55194,6104,10140,11.1
2,"Abilene, TX Metro Area",176656,61924,111552,19008,10180,17.0
3,"Ada, OK Micro Area",38116,59457,25124,4407,10220,17.5
4,"Adrian, MI Micro Area",99263,65484,69488,10011,10300,14.4


### 4.1 Column Engineering

Starting by separating the area name with the state and the area type

In [ ]:
df_demographics['city'] = df_demographics['NAME'].str.split(', ').str[0]
df_demographics['state + area'] = df_demographics['NAME'].str.split(', ').str[1]

df_demographics.head(5)


,NAME,population,median_household_income,pop_25plus,bachelors_count,metropolitan statistical area/micropolitan statistical area,pct_bachelors,city,state + area
0,"Aberdeen, SD Micro Area",42292,70693,28243,6728,10100,23.8,Aberdeen,SD Micro Area
1,"Aberdeen, WA Micro Area",75672,59105,55194,6104,10140,11.1,Aberdeen,WA Micro Area
2,"Abilene, TX Metro Area",176656,61924,111552,19008,10180,17.0,Abilene,TX Metro Area
3,"Ada, OK Micro Area",38116,59457,25124,4407,10220,17.5,Ada,OK Micro Area
4,"Adrian, MI Micro Area",99263,65484,69488,10011,10300,14.4,Adrian,MI Micro Area


In [23]:
df_demographics['state'] = df_demographics['state + area'].str.split(' ').str[0]
df_demographics['area_type'] = df_demographics['state + area'].str.split(' ').str[1] + ' ' + df_demographics['state + area'].str.split(' ').str[2]

df_demographics = df_demographics.drop(columns = ['NAME', 'state + area'])

df_demographics.head(5)

,population,median_household_income,pop_25plus,bachelors_count,metropolitan statistical area/micropolitan statistical area,pct_bachelors,city,state,area_type
0,42292,70693,28243,6728,10100,23.8,Aberdeen,SD,Micro Area
1,75672,59105,55194,6104,10140,11.1,Aberdeen,WA,Micro Area
2,176656,61924,111552,19008,10180,17.0,Abilene,TX,Metro Area
3,38116,59457,25124,4407,10220,17.5,Ada,OK,Micro Area
4,99263,65484,69488,10011,10300,14.4,Adrian,MI,Micro Area


In [ ]:
#Keeping only the 15 areas of our aiports
ville = "Atlanta|Dallas|Denver|Los Angeles|New York|Portland|St. Louis|Raleigh|New Orleans|Fort Myers|Reno|Oklahoma City|Grand Rapids|Bozeman"

etat = "GA|TX|CO|CA|NY|OR|MO|NC|LA|FL|NV|OK|MI|MT|ME"

df_demographics = df_demographics[df_demographics["city"].str.contains(ville, case=False, na=False) & df_demographics["state"].str.contains(etat, case=False, na=False)]

In [ ]:
#Adding an airport_id column to establish the relationship with other table later

df_demographics['airport_id'] = ['ATL', 'BZN', 'RSW', 'DFW', 'DEN', 'GRR', 'LAX', 'MSY', 'JFK', 'OKC', 'PWM', 'PDX', 'RDU', 'RNO', 'STL']

In [26]:
df_demographics

,population,median_household_income,pop_25plus,bachelors_count,metropolitan statistical area/micropolitan statistical area,pct_bachelors,city,state,area_type,airport_id
47,6094752,82625,4073889,1027927,12060,25.2,Atlanta-Sandy Springs-Alpharetta,GA,Metro Area,ATL
108,119685,83434,77076,25951,14580,33.7,Bozeman,MT,Micro Area,BZN
140,772902,69368,583335,108170,15980,18.5,Cape Coral-Fort Myers,FL,Metro Area,RSW
212,7673379,83398,4997613,1207983,19100,24.2,Dallas-Fort Worth-Arlington,TX,Metro Area,DFW
228,2959386,96920,2073751,616405,19740,29.7,Denver-Aurora-Lakewood,CO,Metro Area,DEN
337,1087068,76898,718375,164363,24340,22.9,Grand Rapids-Kentwood,MI,Metro Area,GRR
499,13111917,89105,9107327,2134359,31080,23.4,Los Angeles-Long Beach-Anaheim,CA,Metro Area,LAX
603,1264357,62748,881389,179980,35380,20.4,New Orleans-Metairie,LA,Metro Area,MSY
608,19908595,93610,13959399,3426175,35620,24.5,New York-Newark-Jersey City,NY-NJ-PA,Metro Area,JFK
626,1428923,67963,934060,195681,36420,20.9,Oklahoma City,OK,Metro Area,OKC


### 4.2 Changing the data type

In [ ]:
df_demographics.dtypes
#I don't need to change the data type

population                                                       int64
median_household_income                                          int64
pop_25plus                                                       int64
bachelors_count                                                  int64
metropolitan statistical area/micropolitan statistical area        str
pct_bachelors                                                  float64
city                                                            object
state                                                           object
area_type                                                       object
airport_id                                                         str
dtype: object

### 4.3 Exporting to SQL Server

In [28]:
dtype_map = {col: VARCHAR(100) for col in df_demographics.select_dtypes(include=['string', 'object']).columns}
dtype_map.update({col: INTEGER() for col in df_demographics.select_dtypes(include='int64').columns})

df_demographics.to_sql(name ='demographics', con = con, if_exists = 'replace', index = False, dtype= dtype_map)

-1

# 5. Tail_Number & Aircraft_Model

In [ ]:
df_aircraft_model = pd.read_csv('YOUR FILE', skipinitialspace= True)
df_aircraft_model.head(5)


,CODE,MFR,MODEL,TYPE-ACFT,TYPE-ENG,AC-CAT,BUILD-CERT-IND,NO-ENG,NO-SEATS,AC-WEIGHT,SPEED,TC-DATA-SHEET,TC-DATA-HOLDER,Unnamed: 13
0,0020901,AAR AIRLIFT GROUP INC,UH-60A,6,3,1,0,2,15,CLASS 3,0,NaN,NaN,NaN
1,0030109,EXLINE ACE-C,ACE-C,4,1,1,1,1,1,CLASS 1,82,NaN,NaN,NaN
2,003010D,DELEBAUGH,P,4,1,1,1,1,1,CLASS 1,82,NaN,NaN,NaN
3,003010H,DAL PORTO,BABY ACE D,4,1,1,1,1,1,CLASS 1,82,NaN,NaN,NaN
4,003010P,DUNN,BABY ACE,4,1,1,1,1,1,CLASS 1,82,NaN,NaN,NaN


In [43]:
df_aircraft_model = df_aircraft_model[['CODE', 'MFR', 'MODEL', 'TYPE-ACFT', 'AC-CAT', 'NO-SEATS']]

In [ ]:
df_tail_number = pd.read_csv('YOUR FILE', skipinitialspace= True)
df_tail_number.head(5)


,N-NUMBER,SERIAL NUMBER,MFR MDL CODE,ENG MFR MDL,YEAR MFR,TYPE REGISTRANT,NAME,STREET,STREET2,CITY,...,OTHER NAMES(2),OTHER NAMES(3),OTHER NAMES(4),OTHER NAMES(5),EXPIRATION DATE,UNIQUE ID,KIT MFR,KIT MODEL,MODE S CODE HEX,Unnamed: 34
0,100,5334,7100510,17003.0,1940.0,1.0,BENE MARY D ...,PO BOX 329,NaN,KETCHUM,...,NaN,NaN,NaN,NaN,20270430.0,600060,NaN,NaN,A004B3,NaN
1,10000,10000,2130004,NaN,NaN,7.0,9AT LLC ...,511 WEDGEWOOD AVE,NaN,NASHVILLE,...,NaN,NaN,NaN,NaN,20310831.0,1443200,NaN,NaN,A00725,NaN
2,10001,A28,9601202,67007.0,1928.0,1.0,STOOS ROBERT A ...,PO BOX 1056,NaN,LAKELAND,...,NaN,NaN,NaN,NaN,20290228.0,432072,NaN,NaN,A00726,NaN
3,10004,T18208245,2072738,NaN,NaN,7.0,ETOS AIR LLC ...,PO BOX 288,NaN,NEW LONDON,...,NaN,NaN,NaN,NaN,20290331.0,102879,NaN,NaN,A00729,NaN
4,10006,BG-72,1152020,17026.0,1955.0,1.0,COUTCHES ROBERT HERCULES DBA ...,550 AIRWAY BLVD,NaN,LIVERMORE,...,NaN,NaN,NaN,NaN,20280229.0,480110,NaN,NaN,A0072B,NaN


In [ ]:
#Keeping the useful column
df_tail_number = df_tail_number[['N-NUMBER', 'MFR MDL CODE', 'YEAR MFR', 'TYPE AIRCRAFT', 'UNIQUE ID']]

In [46]:
df_tail_number.head(5)

,N-NUMBER,MFR MDL CODE,YEAR MFR,TYPE AIRCRAFT,UNIQUE ID
0,100,7100510,1940.0,4,600060
1,10000,2130004,NaN,4,1443200
2,10001,9601202,1928.0,4,432072
3,10004,2072738,NaN,4,102879
4,10006,1152020,1955.0,4,480110


### 5.1 Changing the data type

In [ ]:
df_aircraft_model.dtypes

CODE           str
MFR            str
MODEL          str
TYPE-ACFT      str
AC-CAT       int64
NO-SEATS     int64
dtype: object

In [ ]:
df_tail_number['YEAR MFR'] = df_tail_number['YEAR MFR'].astype('Int64')
#Renaming the columns
df_tail_number = df_tail_number.rename(columns={'N-NUMBER' : 'N_NUMBER'})

In [54]:
df_tail_number.dtypes

N_NUMBER                 str
MFR MDL CODE             str
YEAR MFR               Int64
TYPE AIRCRAFT            str
UNIQUE ID              int64
REGISTRATION_NUMBER      str
dtype: object

### 5.2 Column Engineering

The merge between aircraft_model and tail_number is done on the columns CODE and MFR MDL CODE
Also, in the tail_number table, you need to add an N in front of each registration. For example, an aircraft registered as N333DX will be recorded as 333DX in the table

In [50]:
df_tail_number['REGISTRATION_NUMBER'] = "N" + df_tail_number["N_NUMBER"].str.strip()

In [ ]:
#Keeping registration number present in the delay table
valid_tail_numbers = set(df_bts_delay['Tail_Number'].dropna())
df_tail_number = df_tail_number[df_tail_number['REGISTRATION_NUMBER'].isin(valid_tail_numbers)]

In [ ]:
#Keeping the manufacturers that have a matching registration in the delay table
df_aircraft_model = df_aircraft_model[df_aircraft_model['CODE'].isin(df_tail_number['MFR MDL CODE'])]

### 5.3 Exporting to SQL Server

In [62]:
dtype_map = {col: VARCHAR(100) for col in df_aircraft_model.select_dtypes(include=['string', 'object']).columns}
dtype_map.update({col: INTEGER() for col in df_aircraft_model.select_dtypes(include='int64').columns})

df_aircraft_model.to_sql(name ='aircraft_model', con = con, if_exists = 'replace', index = False, dtype= dtype_map)

-1

In [63]:
dtype_map = {col: VARCHAR(100) for col in df_tail_number.select_dtypes(include=['string', 'object']).columns}
dtype_map.update({col: INTEGER() for col in df_tail_number.select_dtypes(include='int64').columns})

df_tail_number.to_sql(name ='tail_number', con = con, if_exists = 'replace', index = False, dtype= dtype_map)

-1

# 6. Meteo

In [ ]:
import openmeteo_requests
import pandas as pd
import requests_cache
from retry_requests import retry
# Setup the Open-Meteo API client with cache and retry on error
cache_session = requests_cache.CachedSession('.cache', expire_after = -1)
retry_session = retry(cache_session, retries = 5, backoff_factor = 0.2)
openmeteo = openmeteo_requests.Client(session = retry_session)

#Adding the ID of the airport
locids = ["ATL", "DFW", "DEN", "LAX", "JFK", "PDX", "STL", "RDU", "MSY", "RSW", "RNO", "OKC", "GRR", "BZN", "PWM"]

# Make sure all required weather variables are listed here
# The order of variables in hourly or daily is important to assign them correctly below
url = "https://archive-api.open-meteo.com/v1/archive"
params = {
	"latitude": [33.6407, 32.8998, 39.8561, 33.9416, 40.6413, 45.5898, 38.7487, 35.8776, 29.9934, 26.5362, 39.4991, 35.3931, 42.8808, 45.7772, 43.6462],
	"longitude": [-84.4277, -97.0403, -104.6737, -118.4085, -73.7781, -122.5951, -90.37, -78.7875, -90.258, -81.7552, -119.7681, -97.6007, -85.5228, -111.1531, -70.3093],
	"start_date": ["2018-12-31", "2018-12-31", "2018-12-31", "2018-12-31", "2018-12-31", "2018-12-31", "2018-12-31", "2018-12-31", "2018-12-31", "2018-12-31", "2018-12-31", "2018-12-31", "2018-12-31", "2018-12-31", "2018-12-31"],
	"end_date": ["2026-01-01", "2026-01-01", "2026-01-01", "2026-01-01", "2026-01-01", "2026-01-01", "2026-01-01", "2026-01-01", "2026-01-01", "2026-01-01", "2026-01-01", "2026-01-01", "2026-01-01", "2026-01-01", "2026-01-01"],
	"hourly": ["temperature_2m", "precipitation", "rain", "snowfall", "snow_depth", "weather_code", "cloud_cover_low", "cloud_cover_mid", "cloud_cover_high", "cloud_cover", "wind_speed_10m", "wind_gusts_10m", "wind_direction_10m"],
	"timezone": "auto",
}
responses = openmeteo.weather_api(url, params = params)

all_dataframes = []

# Process 15 locations
for locid, response in zip(locids, responses):
	print(f"\nCoordinates: {response.Latitude()}°N {response.Longitude()}°E")
	print(f"Elevation: {response.Elevation()} m asl")
	print(f"Timezone: {response.Timezone()}{response.TimezoneAbbreviation()}")
	print(f"Timezone difference to GMT+0: {response.UtcOffsetSeconds()}s")
	
	# Process hourly data. The order of variables needs to be the same as requested.
	hourly = response.Hourly()
	hourly_temperature_2m = hourly.Variables(0).ValuesAsNumpy()
	hourly_precipitation = hourly.Variables(1).ValuesAsNumpy()
	hourly_rain = hourly.Variables(2).ValuesAsNumpy()
	hourly_snowfall = hourly.Variables(3).ValuesAsNumpy()
	hourly_snow_depth = hourly.Variables(4).ValuesAsNumpy()
	hourly_weather_code = hourly.Variables(5).ValuesAsNumpy()
	hourly_cloud_cover_low = hourly.Variables(6).ValuesAsNumpy()
	hourly_cloud_cover_mid = hourly.Variables(7).ValuesAsNumpy()
	hourly_cloud_cover_high = hourly.Variables(8).ValuesAsNumpy()
	hourly_cloud_cover = hourly.Variables(9).ValuesAsNumpy()
	hourly_wind_speed_10m = hourly.Variables(10).ValuesAsNumpy()
	hourly_wind_gusts_10m = hourly.Variables(11).ValuesAsNumpy()
	hourly_wind_direction_10m = hourly.Variables(12).ValuesAsNumpy()
	
	hourly_data = {
		"date": pd.date_range(
			start = pd.to_datetime(hourly.Time(), unit = "s", utc = True),
			end =  pd.to_datetime(hourly.TimeEnd(), unit = "s", utc = True),
			freq = pd.Timedelta(seconds = hourly.Interval()),
			inclusive = "left"
		).tz_convert(response.Timezone().decode())
	}
	
	hourly_data["locid"] = locid
	hourly_data["temperature_2m"] = hourly_temperature_2m
	hourly_data["precipitation"] = hourly_precipitation
	hourly_data["rain"] = hourly_rain
	hourly_data["snowfall"] = hourly_snowfall
	hourly_data["snow_depth"] = hourly_snow_depth
	hourly_data["weather_code"] = hourly_weather_code
	hourly_data["cloud_cover_low"] = hourly_cloud_cover_low
	hourly_data["cloud_cover_mid"] = hourly_cloud_cover_mid
	hourly_data["cloud_cover_high"] = hourly_cloud_cover_high
	hourly_data["cloud_cover"] = hourly_cloud_cover
	hourly_data["wind_speed_10m"] = hourly_wind_speed_10m
	hourly_data["wind_gusts_10m"] = hourly_wind_gusts_10m
	hourly_data["wind_direction_10m"] = hourly_wind_direction_10m
	
	hourly_dataframe = pd.DataFrame(data = hourly_data)
	print("\nHourly data\n", hourly_dataframe)
	all_dataframes.append(hourly_dataframe)

df_meteo = pd.concat(all_dataframes, ignore_index=True)


Coordinates: 33.63795852661133°N -84.4168701171875°E
Elevation: 302.0 m asl
Timezone: b'America/New_York'b'GMT-4'
Timezone difference to GMT+0: -14400s

Hourly data
                            date locid  temperature_2m  precipitation  rain  \
0     2018-12-30 23:00:00-05:00   ATL           11.45            0.5   0.5   
1     2018-12-31 00:00:00-05:00   ATL           11.55            0.0   0.0   
2     2018-12-31 01:00:00-05:00   ATL           11.90            0.0   0.0   
3     2018-12-31 02:00:00-05:00   ATL           12.35            0.0   0.0   
4     2018-12-31 03:00:00-05:00   ATL           12.80            0.0   0.0   
...                         ...   ...             ...            ...   ...   
61411 2026-01-01 18:00:00-05:00   ATL           12.85            0.0   0.0   
61412 2026-01-01 19:00:00-05:00   ATL           11.65            0.0   0.0   
61413 2026-01-01 20:00:00-05:00   ATL           10.85            0.0   0.0   
61414 2026-01-01 21:00:00-05:00   ATL           10.15

In [4]:
df_meteo.head(5)

,date,locid,temperature_2m,precipitation,rain,snowfall,snow_depth,weather_code,cloud_cover_low,cloud_cover_mid,cloud_cover_high,cloud_cover,wind_speed_10m,wind_gusts_10m,wind_direction_10m
0,2018-12-30 23:00:00-05:00,ATL,11.45,0.5,0.5,0.0,0.0,53.0,92.0,33.0,4.0,94.0,13.708390,23.039999,119.931427
1,2018-12-31 00:00:00-05:00,ATL,11.55,0.0,0.0,0.0,0.0,3.0,100.0,17.0,2.0,100.0,13.684735,23.039999,116.564987
2,2018-12-31 01:00:00-05:00,ATL,11.90,0.0,0.0,0.0,0.0,3.0,100.0,37.0,82.0,100.0,12.783802,23.039999,122.347420
3,2018-12-31 02:00:00-05:00,ATL,12.35,0.0,0.0,0.0,0.0,3.0,100.0,38.0,8.0,100.0,14.186923,23.759998,125.706779
4,2018-12-31 03:00:00-05:00,ATL,12.80,0.0,0.0,0.0,0.0,3.0,100.0,20.0,85.0,100.0,15.192682,25.559999,126.326920


### 6.1 Changing the data type

In [ ]:
# Converting date column to str to strip UTC timezone then back to datetime
df_meteo["date"] = pd.to_datetime(df_meteo["date"].astype(str).str[:19])

In [6]:
df_meteo.head(5)

,date,locid,temperature_2m,precipitation,rain,snowfall,snow_depth,weather_code,cloud_cover_low,cloud_cover_mid,cloud_cover_high,cloud_cover,wind_speed_10m,wind_gusts_10m,wind_direction_10m
0,2018-12-30 23:00:00,ATL,11.45,0.5,0.5,0.0,0.0,53.0,92.0,33.0,4.0,94.0,13.708390,23.039999,119.931427
1,2018-12-31 00:00:00,ATL,11.55,0.0,0.0,0.0,0.0,3.0,100.0,17.0,2.0,100.0,13.684735,23.039999,116.564987
2,2018-12-31 01:00:00,ATL,11.90,0.0,0.0,0.0,0.0,3.0,100.0,37.0,82.0,100.0,12.783802,23.039999,122.347420
3,2018-12-31 02:00:00,ATL,12.35,0.0,0.0,0.0,0.0,3.0,100.0,38.0,8.0,100.0,14.186923,23.759998,125.706779
4,2018-12-31 03:00:00,ATL,12.80,0.0,0.0,0.0,0.0,3.0,100.0,20.0,85.0,100.0,15.192682,25.559999,126.326920


### 6.2 Handle the daylight saving time transitions (winter/summer time)

In [ ]:
#Summer time case (duplicates at 1am), I keep the first duplicate
df_meteo = df_meteo.drop_duplicates(subset=['locid', 'date'], keep='first')

In [ ]:
#Winter time case (we jump from 1am to 3am), so I'll create a row for 2am and duplicate the 3am data
df_meteo = (df_meteo.groupby('locid')
        .resample('h', on='date')  # create the 2am row
        .first()  # taking the first entry for each hour
        .bfill()  # backward fill the nulls, duplicating the 3am values
        .reset_index())

### 6.2 Exporting to SQL Server

In [9]:
dtype_map = {col: VARCHAR(100) for col in df_meteo.select_dtypes(include='string').columns}


df_meteo.to_sql(name ='weather', con = con, if_exists = 'replace', index = False, dtype= dtype_map)

-1

# 7. Airline

I create a lookup table to map IATA codes to airline names

In [34]:
airline_names = {
    "AS": "Alaska Airlines",
    "OO": "SkyWest Airlines",
    "DL": "Delta Air Lines",
    "9E": "Endeavor Air",
    "EV": "ExpressJet Airlines",
    "QX": "Horizon Air",
    "MQ": "Envoy Air",
    "YX": "Republic Airways",
    "YV": "Mesa Airlines",
    "AA": "American Airlines",
    "WN": "Southwest Airlines",
    "UA": "United Airlines",
    "B6": "JetBlue Airways",
    "NK": "Spirit Airlines",
    "OH": "PSA Airlines",
    "HA": "Hawaiian Airlines",
    "G4": "Allegiant Air",
    "F9": "Frontier Airlines",
}

df_airline = pd.DataFrame(
    list(airline_names.items()),
    columns=["IATA_CODE_Reporting_Airline", "Airline_Name"]
)



In [35]:
df_airline.head(5)

,IATA_CODE_Reporting_Airline,Airline_Name
0,AS,Alaska Airlines
1,OO,SkyWest Airlines
2,DL,Delta Air Lines
3,9E,Endeavor Air
4,EV,ExpressJet Airlines


### 7.1 Changing the data type

In [36]:
df_airline.dtypes

IATA_CODE_Reporting_Airline    str
Airline_Name                   str
dtype: object

### 7.2 Exporting to SQL Server

In [37]:
dtype_map = {col: VARCHAR(100) for col in df_airline.select_dtypes(include='string').columns}


df_airline.to_sql(name ='airline', con = con, if_exists = 'replace', index = False, dtype= dtype_map)

-1